# CLAUD·IA — Preparar FLEURS para el fine-tuning

Preparamos un conjunto piloto para entrenar la parte de **voz en español → texto en inglés** del Transformer. Cada ejemplo vincula una grabación en español con su traducción de referencia al inglés.

Este notebook prepara y verifica los datos. Las grabaciones personales se reservan para la evaluación final. Esta etapa se puede ejecutar en CPU.

[Conjunto de datos FLEURS de Google](https://huggingface.co/datasets/google/fleurs) · [Artículo sobre FLEURS](https://arxiv.org/abs/2205.12446)

## 1. Instalar las dependencias

En un nuevo notebook de Colab, ejecuta esta celda. Si cambiaron paquetes que ya se habían importado, reinicia el kernel antes de continuar.

In [ ]:
%pip install "datasets==4.8.5" "huggingface-hub>=1.32,<2" soundfile

## 2. Relacionar las frases de los dos idiomas

FLEURS asigna el mismo identificador a una frase y a sus traducciones. Leemos los textos ingleses de `en_us` y los audios en español de `es_419`, manteniendo separadas las particiones `train` y `validation`. La lectura en flujo (*streaming*) permite detenerse después de un número pequeño de ejemplos. Para la primera prueba, conservamos audios de ocho segundos como máximo y una sola grabación por identificador.

In [ ]:
from io import BytesIO

import soundfile as sf
from datasets import Audio as DatasetAudio, load_dataset

dataset_name = "google/fleurs"
sample_rate = 16_000
max_seconds = 8
train_size = 16
validation_size = 4


def collect_pairs(split, count, excluded_ids=()):
    english = load_dataset(dataset_name, "en_us", split=split, streaming=True)
    english = english.select_columns(["id", "raw_transcription"])
    references = {
        int(row["id"]): row["raw_transcription"].strip()
        for row in english
        if row["raw_transcription"] and row["raw_transcription"].strip()
    }

    spanish = load_dataset(dataset_name, "es_419", split=split, streaming=True)
    spanish = spanish.select_columns(["id", "audio", "raw_transcription"])
    spanish = spanish.cast_column("audio", DatasetAudio(decode=False))
    seen = set(excluded_ids)
    pairs = []

    for row in spanish:
        sentence_id = int(row["id"])
        if sentence_id in seen or sentence_id not in references:
            continue

        source = row["raw_transcription"]
        audio_bytes = row["audio"]["bytes"]
        if not source or not source.strip() or not audio_bytes:
            continue

        samples, rate = sf.read(BytesIO(audio_bytes), dtype="float32", always_2d=True)
        if rate != sample_rate:
            raise ValueError(f"Frecuencia inesperada para la frase {sentence_id} : {rate} Hz.")
        if not 0 < len(samples) <= max_seconds * rate:
            continue

        pairs.append({
            "sentence_id": sentence_id,
            "source_text": source.strip(),
            "target_text": references[sentence_id],
            "audio": samples.mean(axis=1),
        })
        seen.add(sentence_id)
        if len(pairs) == count:
            return pairs

    raise RuntimeError(f"Partition {split} : {len(pairs)} paires valides sur {count} demandées.")


train_pairs = collect_pairs("train", train_size)
train_ids = {pair["sentence_id"] for pair in train_pairs}
validation_pairs = collect_pairs("validation", validation_size, excluded_ids=train_ids)
validation_ids = {pair["sentence_id"] for pair in validation_pairs}
assert train_ids.isdisjoint(validation_ids)

print(f"Entraînement : {len(train_pairs)} paires")
print(f"Validation : {len(validation_pairs)} paires")
print("Identifiants communs : 0")

## 3. Escuchar y comprobar un par de datos

Escucha la frase en español y después lee su traducción de referencia al inglés. El texto español mostrado ayuda a comprobar la asociación. Durante el fine-tuning, la entrada será el audio en español y el objetivo será el texto inglés.

In [ ]:
from IPython.display import Audio, display

example = train_pairs[0]
print(f"Frase FLEURS n.º {example['sentence_id']}")
print(f"Español: {example['source_text']}")
print(f"Inglés: {example['target_text']}")
display(Audio(example["audio"], rate=sample_rate))

## 4. Exportar el conjunto piloto

Los archivos WAV y JSONL contienen las entradas y las referencias para el próximo notebook de entrenamiento. El archivo comprimido permite conservar el conjunto si termina la sesión de Colab. Descárgalo desde el panel **Archivos**. Las dos particiones sirven para preparar el flujo de procesamiento y verificar que funciona.

In [ ]:
import json
import shutil
from pathlib import Path

output_dir = Path("fleurs_pilote")

for split, pairs in (("train", train_pairs), ("validation", validation_pairs)):
    split_dir = output_dir / split
    split_dir.mkdir(parents=True, exist_ok=True)
    records = []

    for pair in pairs:
        audio_path = split_dir / f"{pair['sentence_id']}.wav"
        sf.write(audio_path, pair["audio"], sample_rate)
        records.append({
            "sentence_id": pair["sentence_id"],
            "audio_path": audio_path.relative_to(output_dir).as_posix(),
            "source_text": pair["source_text"],
            "target_text": pair["target_text"],
            "sampling_rate": sample_rate,
        })

    with (output_dir / f"{split}.jsonl").open("w", encoding="utf-8") as manifest:
        for record in records:
            print(json.dumps(record, ensure_ascii=False), file=manifest)

archive = shutil.make_archive(output_dir.name, "zip", root_dir=output_dir)
print(f"Archivo comprimido listo: {Path(archive).name}")